In [46]:
import pandas as pd
import numpy as np
import re

# CREATE ROLLING STATS FOR WRESTLERS FOR INDIVDUAL MATCHES, and overall season

In [29]:
# -- read in data
matches = pd.read_csv('matches_updated.csv')
wrestlers = pd.read_csv('wrestlers_updated.csv')

all_wrestlers = pd.read_csv('ALL_WRESTLERS.xls')

teams = pd.read_csv('teams.xls')
duals = pd.read_csv('dual_meets.xls')

In [30]:
matches_sub = matches[
    ~matches["win_type"].isin(["WFOR", "LFOR", "WINJ", "LINJ", "WDQ", "LDQ"])
].copy()

In [31]:
# def compute_match_level_stats(df):
#     """
#     Compute lagged rolling statistics for wrestlers.
#     Features:
#     - avg_points_scored: mean points scored (excludes falls)
#     - std_point_diff: standard deviation of point differential (excludes falls)
#     - win_rate, bonus_rate, pin_count, avg_opponent_rank, avg_point_diff
#     """
#     df = df.copy()
#     df["event_date"] = pd.to_datetime(df["event_date"])
#     df = df.sort_values("event_date").reset_index(drop=True)
    
#     # Create unique match ID for proper merging
#     df["_match_id"] = df.index
    
#     # Helper: extract base win type
#     def get_base_win_type(win_type):
#         wt = win_type.upper().strip()
#         if "FALL" in wt:
#             return "FALL"
#         elif "TF" in wt:
#             return "TF"
#         elif "MD" in wt:
#             return "MD"
#         return None
    
#     # Bonus win types
#     bonus_types = {"TF", "FALL", "MD"}
    
#     # Parse points
#     def extract_points(result, win_type):
#         if "FALL" in win_type:
#             return None, None, True, False
        
#         is_tf = "TF" in win_type.upper()
        
#         try:
#             parts = result.split()
#             scores = [int(x) for x in parts if x.isdigit()]
#             if len(scores) >= 2:
#                 return scores[0], scores[1], False, is_tf
#             return None, None, False, False
#         except:
#             return None, None, False, False
    
#     # Extract scores
#     scores_data = df.apply(
#         lambda r: extract_points(r["Result"], r["win_type"]), 
#         axis=1
#     )
#     df["winner_score"] = [s[0] for s in scores_data]
#     df["loser_score"] = [s[1] for s in scores_data]
#     df["is_fall"] = [s[2] for s in scores_data]
#     df["is_tf"] = [s[3] for s in scores_data]
    
#     # Build long format
#     records = []
#     for _, row in df.iterrows():
#         base_type = get_base_win_type(row["win_type"])
#         is_bonus_type = base_type in bonus_types if base_type else False
        
#         skip_points = row["is_fall"]
        
#         # Home wrestler
#         home_won = row["home_win"]
#         home_bonus = home_won and is_bonus_type
#         home_pin = home_won and base_type == "FALL"
        
#         if not skip_points and row["winner_score"] is not None:
#             if home_won:
#                 home_points = row["winner_score"]
#                 home_point_diff = 15 if row["is_tf"] else row["winner_score"] - row["loser_score"]
#             else:
#                 home_points = row["loser_score"]
#                 home_point_diff = -15 if row["is_tf"] else row["loser_score"] - row["winner_score"]
#         else:
#             home_points = None
#             home_point_diff = None
        
#         records.append({
#             "_match_id": row["_match_id"],
#             "event_date": row["event_date"],
#             "wrestler": row["home_name"],
#             "opponent": row["away_name"],
#             "wrestler_rank": row["home_rank"],
#             "opponent_rank": row["away_rank"],
#             "won": home_won,
#             "bonus_win": home_bonus,
#             "pin_win": home_pin,
#             "point_diff": home_point_diff,
#             "points_scored": home_points,
#             "side": "home"
#         })
        
#         # Away wrestler
#         away_won = not home_won
#         away_bonus = away_won and is_bonus_type
#         away_pin = away_won and base_type == "FALL"
        
#         if not skip_points and row["winner_score"] is not None:
#             if away_won:
#                 away_points = row["winner_score"]
#                 away_point_diff = 15 if row["is_tf"] else row["winner_score"] - row["loser_score"]
#             else:
#                 away_points = row["loser_score"]
#                 away_point_diff = -15 if row["is_tf"] else row["loser_score"] - row["winner_score"]
#         else:
#             away_points = None
#             away_point_diff = None
        
#         records.append({
#             "_match_id": row["_match_id"],
#             "event_date": row["event_date"],
#             "wrestler": row["away_name"],
#             "opponent": row["home_name"],
#             "wrestler_rank": row["away_rank"],
#             "opponent_rank": row["home_rank"],
#             "won": away_won,
#             "bonus_win": away_bonus,
#             "pin_win": away_pin,
#             "point_diff": away_point_diff,
#             "points_scored": away_points,
#             "side": "away"
#         })
    
#     long_df = pd.DataFrame(records)
#     long_df = long_df.sort_values(["wrestler", "event_date", "_match_id"]).reset_index(drop=True)
    
#     # Rolling stats
#     long_df["matches_wrestled"] = long_df.groupby("wrestler").cumcount()
    
#     long_df["win_rate"] = (
#         long_df.groupby("wrestler")["won"]
#         .transform(lambda x: x.shift().expanding().mean())
#     )
    
#     long_df["loss_rate"] = 1 - long_df["win_rate"]
    
#     long_df["bonus_rate"] = (
#         long_df.groupby("wrestler")["bonus_win"]
#         .transform(lambda x: x.shift().expanding().mean())
#     )
    
#     long_df["pin_count"] = (
#         long_df.groupby("wrestler")["pin_win"]
#         .transform(lambda x: x.shift().cumsum())
#     )
    
#     long_df["avg_opponent_rank"] = (
#         long_df.groupby("wrestler")["opponent_rank"]
#         .transform(lambda x: x.shift().expanding().mean())
#     )
    
#     def expanding_mean_non_none(series):
#         shifted = series.shift()
#         result = pd.Series(index=series.index, dtype=float)
#         for i in range(len(series)):
#             prior = shifted.iloc[:i+1].dropna()
#             result.iloc[i] = prior.mean() if len(prior) > 0 else 0.0
#         return result
    
#     long_df["avg_point_diff"] = (
#         long_df.groupby("wrestler")["point_diff"]
#         .transform(expanding_mean_non_none)
#     )
    
#     long_df["avg_points_scored"] = (
#         long_df.groupby("wrestler")["points_scored"]
#         .transform(expanding_mean_non_none)
#     )
    
#     def expanding_std_non_none(series):
#         shifted = series.shift()
#         result = pd.Series(index=series.index, dtype=float)
#         for i in range(len(series)):
#             prior = shifted.iloc[:i+1].dropna()
#             result.iloc[i] = prior.std() if len(prior) > 1 else 0.0
#         return result
    
#     long_df["std_point_diff"] = (
#         long_df.groupby("wrestler")["point_diff"]
#         .transform(expanding_std_non_none)
#     )
    
#     # Fill NaNs
#     stat_cols = [
#         "win_rate", "loss_rate", "bonus_rate", "pin_count", 
#         "avg_opponent_rank", "avg_point_diff", "avg_points_scored", "std_point_diff"
#     ]
#     long_df[stat_cols] = long_df[stat_cols].fillna(0)
    
#     # Merge back
#     df_out = df.copy()
    
#     home_stats = long_df[long_df["side"] == "home"].copy()
#     away_stats = long_df[long_df["side"] == "away"].copy()
    
#     home_stats = home_stats.rename(columns={
#         "wrestler": "home_name",
#         "matches_wrestled": "home_matches_wrestled",
#         "win_rate": "home_win_rate",
#         "loss_rate": "home_loss_rate",
#         "bonus_rate": "home_bonus_rate",
#         "pin_count": "home_pin_count",
#         "avg_opponent_rank": "home_avg_opponent_rank",
#         "avg_point_diff": "home_avg_point_diff",
#         "avg_points_scored": "home_avg_points_scored",
#         "std_point_diff": "home_std_point_diff"
#     })
    
#     away_stats = away_stats.rename(columns={
#         "wrestler": "away_name",
#         "matches_wrestled": "away_matches_wrestled",
#         "win_rate": "away_win_rate",
#         "loss_rate": "away_loss_rate",
#         "bonus_rate": "away_bonus_rate",
#         "pin_count": "away_pin_count",
#         "avg_opponent_rank": "away_avg_opponent_rank",
#         "avg_point_diff": "away_avg_point_diff",
#         "avg_points_scored": "away_avg_points_scored",
#         "std_point_diff": "away_std_point_diff"
#     })
    
#     df_out = df_out.merge(
#         home_stats[[
#             "_match_id", "home_matches_wrestled", "home_win_rate", "home_loss_rate",
#             "home_bonus_rate", "home_pin_count", "home_avg_opponent_rank",
#             "home_avg_point_diff", "home_avg_points_scored", "home_std_point_diff"
#         ]],
#         on="_match_id", how="left"
#     )
    
#     df_out = df_out.merge(
#         away_stats[[
#             "_match_id", "away_matches_wrestled", "away_win_rate", "away_loss_rate",
#             "away_bonus_rate", "away_pin_count", "away_avg_opponent_rank",
#             "away_avg_point_diff", "away_avg_points_scored", "away_std_point_diff"
#         ]],
#         on="_match_id", how="left"
#     )
    
#     df_out = df_out.drop(columns=["_match_id", "winner_score", "loser_score", "is_fall", "is_tf"])
    
#     return df_out

In [32]:
def compute_match_level_stats(df):
    """
    Compute lagged rolling statistics for wrestlers using unique wrestler_id.
    """
    df = df.copy()
    df["event_date"] = pd.to_datetime(df["event_date"])
    df = df.sort_values("event_date").reset_index(drop=True)
    
    # Create unique match ID
    df["_match_id"] = df.index
    
    # Helper: extract base win type
    def get_base_win_type(win_type):
        if pd.isna(win_type):
            return None
        wt = str(win_type).upper().strip()
        if "FALL" in wt:
            return "FALL"
        elif "TF" in wt:
            return "TF"
        elif "MD" in wt:
            return "MD"
        return None
    
    # Bonus win types
    bonus_types = {"TF", "FALL", "MD"}
    
    # Parse points
    def extract_points(result, win_type):
        if pd.isna(win_type) or "FALL" in str(win_type).upper():
            return None, None, True, False
        
        is_tf = "TF" in str(win_type).upper()
        
        if pd.isna(result):
            return None, None, False, False
            
        try:
            parts = str(result).split()
            scores = [int(x) for x in parts if x.isdigit()]
            if len(scores) >= 2:
                return scores[0], scores[1], False, is_tf
            return None, None, False, False
        except:
            return None, None, False, False
    
    # Extract scores
    scores_data = df.apply(
        lambda r: extract_points(r["Result"], r["win_type"]), 
        axis=1
    )
    df["winner_score"] = [s[0] for s in scores_data]
    df["loser_score"] = [s[1] for s in scores_data]
    df["is_fall"] = [s[2] for s in scores_data]
    df["is_tf"] = [s[3] for s in scores_data]
    
    # Build long format with unique identifier (wrestler_id + name as backup)
    records = []
    for _, row in df.iterrows():
        base_type = get_base_win_type(row["win_type"])
        is_bonus_type = base_type in bonus_types if base_type else False
        
        skip_points = row["is_fall"]
        
        # Create unique keys for wrestlers
        home_key = f"{row['home_wrestler_id']}_{row['home_name']}" if pd.notna(row['home_wrestler_id']) else row['home_name']
        away_key = f"{row['away_wrestler_id']}_{row['away_name']}" if pd.notna(row['away_wrestler_id']) else row['away_name']
        
        # Home wrestler
        home_won = row["home_win"]
        home_bonus = home_won and is_bonus_type
        home_pin = home_won and base_type == "FALL"
        
        if not skip_points and row["winner_score"] is not None:
            if home_won:
                home_points = row["winner_score"]
                home_point_diff = 15 if row["is_tf"] else row["winner_score"] - row["loser_score"]
            else:
                home_points = row["loser_score"]
                home_point_diff = -15 if row["is_tf"] else row["loser_score"] - row["winner_score"]
        else:
            home_points = None
            home_point_diff = None
        
        records.append({
            "_match_id": row["_match_id"],
            "event_date": row["event_date"],
            "wrestler_id": row["home_wrestler_id"],
            "wrestler_name": row["home_name"],
            "wrestler_key": home_key,
            "opponent": row["away_name"],
            "wrestler_rank": row["home_rank"],
            "opponent_rank": row["away_rank"],
            "won": home_won,
            "bonus_win": home_bonus,
            "pin_win": home_pin,
            "point_diff": home_point_diff,
            "points_scored": home_points,
            "side": "home"
        })
        
        # Away wrestler
        away_won = not home_won
        away_bonus = away_won and is_bonus_type
        away_pin = away_won and base_type == "FALL"
        
        if not skip_points and row["winner_score"] is not None:
            if away_won:
                away_points = row["winner_score"]
                away_point_diff = 15 if row["is_tf"] else row["winner_score"] - row["loser_score"]
            else:
                away_points = row["loser_score"]
                away_point_diff = -15 if row["is_tf"] else row["loser_score"] - row["winner_score"]
        else:
            away_points = None
            away_point_diff = None
        
        records.append({
            "_match_id": row["_match_id"],
            "event_date": row["event_date"],
            "wrestler_id": row["away_wrestler_id"],
            "wrestler_name": row["away_name"],
            "wrestler_key": away_key,
            "opponent": row["home_name"],
            "wrestler_rank": row["away_rank"],
            "opponent_rank": row["home_rank"],
            "won": away_won,
            "bonus_win": away_bonus,
            "pin_win": away_pin,
            "point_diff": away_point_diff,
            "points_scored": away_points,
            "side": "away"
        })
    
    long_df = pd.DataFrame(records)
    long_df = long_df.sort_values(["wrestler_key", "event_date", "_match_id"]).reset_index(drop=True)
    
    # Rolling stats per wrestler (using wrestler_key as unique identifier)
    long_df["matches_wrestled"] = long_df.groupby("wrestler_key").cumcount()
    
    long_df["win_rate"] = (
        long_df.groupby("wrestler_key")["won"]
        .transform(lambda x: x.shift().expanding().mean())
    )
    
    long_df["loss_rate"] = 1 - long_df["win_rate"]
    
    long_df["bonus_rate"] = (
        long_df.groupby("wrestler_key")["bonus_win"]
        .transform(lambda x: x.shift().expanding().mean())
    )
    
    long_df["pin_count"] = (
        long_df.groupby("wrestler_key")["pin_win"]
        .transform(lambda x: x.shift().cumsum())
    )
    
    long_df["avg_opponent_rank"] = (
        long_df.groupby("wrestler_key")["opponent_rank"]
        .transform(lambda x: x.shift().expanding().mean())
    )
    
    def expanding_mean_non_none(series):
        shifted = series.shift()
        result = pd.Series(index=series.index, dtype=float)
        for i in range(len(series)):
            prior = shifted.iloc[:i+1].dropna()
            result.iloc[i] = prior.mean() if len(prior) > 0 else 0.0
        return result
    
    long_df["avg_point_diff"] = (
        long_df.groupby("wrestler_key")["point_diff"]
        .transform(expanding_mean_non_none)
    )
    
    long_df["avg_points_scored"] = (
        long_df.groupby("wrestler_key")["points_scored"]
        .transform(expanding_mean_non_none)
    )
    
    def expanding_std_non_none(series):
        shifted = series.shift()
        result = pd.Series(index=series.index, dtype=float)
        for i in range(len(series)):
            prior = shifted.iloc[:i+1].dropna()
            result.iloc[i] = prior.std() if len(prior) > 1 else 0.0
        return result
    
    long_df["std_point_diff"] = (
        long_df.groupby("wrestler_key")["point_diff"]
        .transform(expanding_std_non_none)
    )
    
    # Fill NaNs
    stat_cols = [
        "win_rate", "loss_rate", "bonus_rate", "pin_count", 
        "avg_opponent_rank", "avg_point_diff", "avg_points_scored", "std_point_diff"
    ]
    long_df[stat_cols] = long_df[stat_cols].fillna(0)
    
    # Merge back to match-level using wrestler_name and side
    df_out = df.copy()
    
    # Create merge keys
    home_for_merge = long_df[long_df["side"] == "home"].copy()
    away_for_merge = long_df[long_df["side"] == "away"].copy()
    
    # Merge home stats using match_id and home_name
    df_out = df_out.merge(
        home_for_merge[[
            "_match_id", "wrestler_name", "matches_wrestled", "win_rate", "loss_rate",
            "bonus_rate", "pin_count", "avg_opponent_rank", "avg_point_diff", 
            "avg_points_scored", "std_point_diff"
        ]].rename(columns={
            "wrestler_name": "home_name",
            "matches_wrestled": "home_matches_wrestled",
            "win_rate": "home_win_rate",
            "loss_rate": "home_loss_rate",
            "bonus_rate": "home_bonus_rate",
            "pin_count": "home_pin_count",
            "avg_opponent_rank": "home_avg_opponent_rank",
            "avg_point_diff": "home_avg_point_diff",
            "avg_points_scored": "home_avg_points_scored",
            "std_point_diff": "home_std_point_diff"
        }),
        on=["_match_id", "home_name"],
        how="left"
    )
    
    # Merge away stats
    df_out = df_out.merge(
        away_for_merge[[
            "_match_id", "wrestler_name", "matches_wrestled", "win_rate", "loss_rate",
            "bonus_rate", "pin_count", "avg_opponent_rank", "avg_point_diff", 
            "avg_points_scored", "std_point_diff"
        ]].rename(columns={
            "wrestler_name": "away_name",
            "matches_wrestled": "away_matches_wrestled",
            "win_rate": "away_win_rate",
            "loss_rate": "away_loss_rate",
            "bonus_rate": "away_bonus_rate",
            "pin_count": "away_pin_count",
            "avg_opponent_rank": "away_avg_opponent_rank",
            "avg_point_diff": "away_avg_point_diff",
            "avg_points_scored": "away_avg_points_scored",
            "std_point_diff": "away_std_point_diff"
        }),
        on=["_match_id", "away_name"],
        how="left"
    )
    
    # Drop temporary columns
    df_out = df_out.drop(columns=["_match_id", "winner_score", "loser_score", "is_fall", "is_tf"])
    
    return df_out

In [33]:
# -- test on sample
wrestlers = ["Drake Ayala", "Ben Davino", "Sergio Vega", "Brock Hardy"]
sample = matches_sub.query('home_name in @wrestlers or away_name in @wrestlers')
rolling_sample = compute_match_level_stats(sample)

/tmp/ipykernel_3505/3410227953.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_df[stat_cols] = long_df[stat_cols].fillna(0)


In [34]:
rolling_sample

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,...,home_std_point_diff,away_matches_wrestled,away_win_rate,away_loss_rate,away_bonus_rate,away_pin_count,away_avg_opponent_rank,away_avg_point_diff,away_avg_points_scored,away_std_point_diff
0,275,133,2025-11-06,196.0,Drake Ayala,6.0,1005.0,Trayce Eckman,74.0,True,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,272,141,2025-11-07,217.0,Brock Hardy,3.0,960.0,Braden Basile,32.0,True,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
2,273,141,2025-11-07,592.0,Sergio Vega,1.0,257.0,Jack Consiglio,31.0,True,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
3,253,133,2025-11-09,1226.0,Ethan Lipsey,158.0,443.0,Ben Davino,3.0,False,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,219,133,2025-11-15,196.0,Drake Ayala,6.0,928.0,Kade Moore,29.0,True,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,435,141,2026-02-15,306.0,Tom Crook,40.0,592.0,Sergio Vega,2.0,False,...,0.000000,13,1.000000,0.000000,0.461538,2.0,35.153846,6.818182,8.090909,5.173358
60,451,133,2026-02-15,345.0,Braxton Brown,23.0,443.0,Ben Davino,4.0,False,...,0.000000,18,0.944444,0.055556,0.611111,1.0,65.222222,9.941176,13.294118,5.899900
61,564,141,2026-02-21,872.0,Haiden Drury,31.0,217.0,Brock Hardy,3.0,False,...,0.000000,17,0.764706,0.235294,0.588235,4.0,35.647059,6.250000,11.000000,10.331989
62,550,141,2026-02-22,592.0,Sergio Vega,2.0,197.0,Kale Petersen,43.0,True,...,4.972652,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000


In [35]:
# -- test
for w in wrestlers:
    print(f"================================{w}================================")
    temp = rolling_sample.query(
        'home_name == @w or away_name == @w'
    )

    print(f"*************Home*************")
    display(temp[[
        "dual_id", "event_date", "home_name", "home_rank", "away_name", "away_rank", "home_win", "win_type", "Result", "home_matches_wrestled", "home_win_rate", "home_bonus_rate", "home_pin_count", "home_avg_opponent_rank", "home_avg_point_diff",
        'home_avg_points_scored', 'home_std_point_diff'
    ]])
    
    print(); print()
    print(f"*************AWAY*************")
    display(temp[[
        "dual_id", "event_date", "home_name", "home_rank", "away_name", "away_rank", "home_win", "win_type", "Result", "away_matches_wrestled", "away_win_rate", "away_bonus_rate", "away_pin_count", "away_avg_opponent_rank", "away_avg_point_diff",
        'away_avg_points_scored', 'away_std_point_diff'
    ]])


    print(); print(); print()

================================Drake Ayala================================
*************Home*************


,dual_id,event_date,home_name,home_rank,away_name,away_rank,home_win,win_type,Result,home_matches_wrestled,home_win_rate,home_bonus_rate,home_pin_count,home_avg_opponent_rank,home_avg_point_diff,home_avg_points_scored,home_std_point_diff
0,275,2025-11-06,Drake Ayala,6.0,Trayce Eckman,74.0,True,WTF,WTF5 19 - 4 4:28,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,219,2025-11-15,Drake Ayala,6.0,Kade Moore,29.0,True,WFALL,WFALL 3:57,1,1.000000,1.000000,0.0,74.000000,15.000000,19.000000,0.000000
10,206,2025-11-15,Drake Ayala,6.0,Lucas Byrd,1.0,False,LDEC,LDEC 7 - 2,2,1.000000,1.000000,1.0,51.500000,15.000000,19.000000,0.000000
16,188,2025-11-16,Ben Davino,3.0,Drake Ayala,6.0,True,WDEC,WDEC 10 - 4,4,1.000000,1.000000,0.0,86.250000,13.500000,17.000000,3.000000
17,187,2025-11-16,Drake Ayala,6.0,Ronnie Ramirez,28.0,True,WSV,WSV-1 7 - 4,4,0.500000,0.500000,1.0,26.750000,1.333333,8.333333,11.846237
19,178,2025-11-21,Drake Ayala,6.0,Evan Tallmadge,50.0,True,WMD,WMD 13 - 4,5,0.600000,0.400000,1.0,27.000000,1.750000,8.000000,9.708244
21,162,2025-11-30,Evan Frost,5.0,Drake Ayala,6.0,True,WDEC,WDEC 11 - 5,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
33,29,2026-01-09,Drake Ayala,6.0,Zan Fugitt,10.0,False,LDEC,LDEC 6 - 5,7,0.571429,0.428571,1.0,27.142857,1.666667,8.333333,8.891944
37,324,2026-01-16,Drake Ayala,9.0,Marcus Blaze,4.0,False,LDEC,LDEC 4 - 2,8,0.500000,0.375000,1.0,25.000000,1.285714,7.857143,8.179533
40,373,2026-01-23,Jacob Van Dee,11.0,Drake Ayala,8.0,False,LDEC,LDEC 12 - 6,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000




*************AWAY*************


,dual_id,event_date,home_name,home_rank,away_name,away_rank,home_win,win_type,Result,away_matches_wrestled,away_win_rate,away_bonus_rate,away_pin_count,away_avg_opponent_rank,away_avg_point_diff,away_avg_points_scored,away_std_point_diff
0,275,2025-11-06,Drake Ayala,6.0,Trayce Eckman,74.0,True,WTF,WTF5 19 - 4 4:28,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,219,2025-11-15,Drake Ayala,6.0,Kade Moore,29.0,True,WFALL,WFALL 3:57,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
10,206,2025-11-15,Drake Ayala,6.0,Lucas Byrd,1.0,False,LDEC,LDEC 7 - 2,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
16,188,2025-11-16,Ben Davino,3.0,Drake Ayala,6.0,True,WDEC,WDEC 10 - 4,3,0.666667,0.666667,1.0,34.666667,5.000000,10.500000,14.142136
17,187,2025-11-16,Drake Ayala,6.0,Ronnie Ramirez,28.0,True,WSV,WSV-1 7 - 4,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
19,178,2025-11-21,Drake Ayala,6.0,Evan Tallmadge,50.0,True,WMD,WMD 13 - 4,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
21,162,2025-11-30,Evan Frost,5.0,Drake Ayala,6.0,True,WDEC,WDEC 11 - 5,6,0.666667,0.500000,1.0,30.833333,3.200000,9.000000,9.011104
33,29,2026-01-09,Drake Ayala,6.0,Zan Fugitt,10.0,False,LDEC,LDEC 6 - 5,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
37,324,2026-01-16,Drake Ayala,9.0,Marcus Blaze,4.0,False,LDEC,LDEC 4 - 2,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
40,373,2026-01-23,Jacob Van Dee,11.0,Drake Ayala,8.0,False,LDEC,LDEC 12 - 6,9,0.444444,0.333333,1.0,22.666667,0.875000,7.125000,7.661359





================================Ben Davino================================
*************Home*************


...

,dual_id,event_date,home_name,home_rank,away_name,away_rank,home_win,win_type,Result,away_matches_wrestled,away_win_rate,away_bonus_rate,away_pin_count,away_avg_opponent_rank,away_avg_point_diff,away_avg_points_scored,away_std_point_diff
2,273,2025-11-07,Sergio Vega,1.0,Jack Consiglio,31.0,True,WTF,WTF5 16 - 1 6:21,0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
7,222,2025-11-15,Sergio Vega,1.0,Joshua Saunders,46.0,True,WDEC,WDEC 5 - 0,0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
8,209,2025-11-15,Sergio Vega,1.0,Ryan Jack,17.0,True,WDEC,WDEC 2 - 0,0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
12,190,2025-11-16,Sergio Vega,1.0,Brock Hardy,3.0,True,WMD,WMD 13 - 2,3,1.0,1.000000,1.0,38.666667,15.000000,19.000000,0.000000
15,187,2025-11-16,Nasir Bailey,12.0,Sergio Vega,1.0,False,LSV,LSV-1 3 - 0,4,1.0,0.500000,0.0,24.250000,8.250000,9.000000,5.852350
20,173,2025-11-23,Pierson Manville,41.0,Sergio Vega,1.0,False,LDEC,LDEC 8 - 3,5,1.0,0.400000,0.0,21.800000,7.200000,7.800000,5.585696
26,117,2025-12-14,Jordan Titus,54.0,Sergio Vega,1.0,False,LDEC,LDEC 5 - 1,6,1.0,0.333333,0.0,25.000000,6.833333,7.833333,5.076088
28,60,2025-12-21,Brock Hardy,3.0,Sergio Vega,1.0,False,LFALL,LFALL 1:47,7,1.0,0.285714,0.0,29.142857,6.428571,7.428571,4.755949
36,293,2026-01-11,Sergio Vega,1.0,Tyler Wells,39.0,True,WDEC,WDEC 3 - 1,0,0.0,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
38,378,2026-01-23,Easton Hilton,55.0,Sergio Vega,2.0,False,LFALL,LFALL 1:31,9,1.0,0.333333,1.0,27.333333,5.875000,6.875000,4.673252





================================Brock Hardy================================
*************Home*************


,dual_id,event_date,home_name,home_rank,away_name,away_rank,home_win,win_type,Result,home_matches_wrestled,home_win_rate,home_bonus_rate,home_pin_count,home_avg_opponent_rank,home_avg_point_diff,home_avg_points_scored,home_std_point_diff
1,272,2025-11-07,Brock Hardy,3.0,Braden Basile,32.0,True,WTF,WTF5 18 - 2 6:16,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
5,220,2025-11-15,Brock Hardy,3.0,Carter Bailey,44.0,True,WFALL,WFALL 6:11,1,1.000000,1.000000,0.0,32.000000,15.000000,18.000000,0.000000
11,207,2025-11-15,Brock Hardy,3.0,Eren Sement,40.0,True,WTF,WTF5 20 - 5 7:00,2,1.000000,1.000000,1.0,38.000000,15.000000,18.000000,0.000000
12,190,2025-11-16,Sergio Vega,1.0,Brock Hardy,3.0,True,WMD,WMD 13 - 2,3,1.000000,0.333333,0.0,31.333333,7.333333,7.666667,6.806859
14,189,2025-11-16,Jesse Mendez,2.0,Brock Hardy,3.0,True,WDEC,WDEC 4 - 1,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
23,148,2025-12-05,Brock Hardy,3.0,Khimari Manns,90.0,True,WFALL,WFALL 2:32,5,0.600000,0.600000,1.0,23.800000,4.000000,10.250000,13.114877
24,149,2025-12-05,Zeke Seltzer,62.0,Brock Hardy,3.0,False,LTF,LTF5 19 - 4 5:40,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
27,99,2025-12-19,Brock Hardy,3.0,Luke Simcox,26.0,True,WDEC,WDEC 5 - 0,7,0.714286,0.714286,2.0,38.714286,6.200000,12.000000,12.377399
28,60,2025-12-21,Brock Hardy,3.0,Sergio Vega,1.0,False,LFALL,LFALL 1:47,8,0.750000,0.625000,2.0,37.125000,6.000000,10.833333,11.081516
31,48,2026-01-03,Brock Hardy,3.0,Cory Land,16.0,True,WDEC,WDEC 5 - 1,9,0.666667,0.555556,2.0,33.111111,6.000000,10.833333,11.081516




*************AWAY*************


,dual_id,event_date,home_name,home_rank,away_name,away_rank,home_win,win_type,Result,away_matches_wrestled,away_win_rate,away_bonus_rate,away_pin_count,away_avg_opponent_rank,away_avg_point_diff,away_avg_points_scored,away_std_point_diff
1,272,2025-11-07,Brock Hardy,3.0,Braden Basile,32.0,True,WTF,WTF5 18 - 2 6:16,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
5,220,2025-11-15,Brock Hardy,3.0,Carter Bailey,44.0,True,WFALL,WFALL 6:11,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
11,207,2025-11-15,Brock Hardy,3.0,Eren Sement,40.0,True,WTF,WTF5 20 - 5 7:00,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
12,190,2025-11-16,Sergio Vega,1.0,Brock Hardy,3.0,True,WMD,WMD 13 - 2,3,1.000000,1.000000,1.0,38.666667,15.000000,19.000000,0.000000
14,189,2025-11-16,Jesse Mendez,2.0,Brock Hardy,3.0,True,WDEC,WDEC 4 - 1,4,0.750000,0.750000,1.0,29.250000,6.333333,13.333333,15.011107
23,148,2025-12-05,Brock Hardy,3.0,Khimari Manns,90.0,True,WFALL,WFALL 2:32,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
24,149,2025-12-05,Zeke Seltzer,62.0,Brock Hardy,3.0,False,LTF,LTF5 19 - 4 5:40,6,0.666667,0.666667,2.0,34.833333,4.000000,10.250000,13.114877
27,99,2025-12-19,Brock Hardy,3.0,Luke Simcox,26.0,True,WDEC,WDEC 5 - 0,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
28,60,2025-12-21,Brock Hardy,3.0,Sergio Vega,1.0,False,LFALL,LFALL 1:47,7,1.000000,0.285714,0.0,29.142857,6.428571,7.428571,4.755949
31,48,2026-01-03,Brock Hardy,3.0,Cory Land,16.0,True,WDEC,WDEC 5 - 1,0,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000


In [36]:
# -- Run function on all matches
matches_w_stats = compute_match_level_stats(matches_sub)
matches_w_stats

/tmp/ipykernel_3505/3410227953.py:199: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  long_df[stat_cols] = long_df[stat_cols].fillna(0)


,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,...,home_std_point_diff,away_matches_wrestled,away_win_rate,away_loss_rate,away_bonus_rate,away_pin_count,away_avg_opponent_rank,away_avg_point_diff,away_avg_points_scored,away_std_point_diff
0,280,141,2025-11-01,267.0,Raymond Adams,160.0,825.0,John Hildebrandt,98.0,False,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
1,281,157,2025-11-01,840.0,Jonathan Ley,46.0,1038.0,Vince Bouzakis,82.0,False,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
2,281,165,2025-11-01,841.0,Dylan Elmore,34.0,298.0,Dylan Evans,22.0,False,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
3,281,174,2025-11-01,842.0,Danny Wask,6.0,300.0,Luca Augustine,21.0,True,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
4,281,184,2025-11-01,1256.0,Daniel Williams,61.0,301.0,Chase Kranitz,38.0,False,...,0.000000,0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5627,547,157,2026-02-23,1448.0,Phil Cuttino,212.0,822.0,Raymond Cmil,205.0,False,...,0.000000,6,0.000000,1.000000,0.000000,0.0,106.500000,-9.800000,1.800000,6.300794
5628,547,174,2026-02-23,399.0,Joshua Roe,237.0,814.0,Beau Lewis,158.0,False,...,3.725289,12,0.166667,0.833333,0.166667,1.0,100.666667,-3.909091,4.636364,7.687061
5629,547,184,2026-02-23,400.0,Reed Douglass,72.0,815.0,Andrew Barford,162.0,True,...,10.122881,11,0.272727,0.727273,0.181818,1.0,111.818182,-6.300000,6.800000,9.499123
5630,547,197,2026-02-23,911.0,Toler Hornick,207.0,816.0,Toby Schoffstall,74.0,False,...,6.976150,8,0.500000,0.500000,0.250000,1.0,98.250000,-2.714286,4.142857,8.400680


In [37]:
# ============================================
# TEST CASE 1: Drake Ayala (Last match vs Jax Forrest)
# ============================================
print("="*60)
print("TEST CASE 1: Drake Ayala")
print("="*60)

drake_last = matches_w_stats.query('home_name == "Drake Ayala" or away_name == "Drake Ayala"').iloc[-1]

print("Expected vs Actual for Drake Ayala:")
print(f"win_rate:           Expected 0.600000 | Actual {drake_last['home_win_rate' if drake_last['home_name'] == 'Drake Ayala' else 'away_win_rate']:.6f}")
print(f"bonus_rate:         Expected 0.466667 | Actual {drake_last['home_bonus_rate' if drake_last['home_name'] == 'Drake Ayala' else 'away_bonus_rate']:.6f}")
print(f"pin_count:          Expected 1.0       | Actual {drake_last['home_pin_count' if drake_last['home_name'] == 'Drake Ayala' else 'away_pin_count']:.6f}")
print(f"avg_opponent_rank:  Expected 30.866667 | Actual {drake_last['home_avg_opponent_rank' if drake_last['home_name'] == 'Drake Ayala' else 'away_avg_opponent_rank']:.6f}")
print(f"avg_point_diff:     Expected 5.071429  | Actual {drake_last['home_avg_point_diff' if drake_last['home_name'] == 'Drake Ayala' else 'away_avg_point_diff']:.6f}")
print(f"avg_points_scored:  Expected 10.928571 | Actual {drake_last['home_avg_points_scored' if drake_last['home_name'] == 'Drake Ayala' else 'away_avg_points_scored']:.6f}")
print(f"std_point_diff:     Expected 8.757101  | Actual {drake_last['home_std_point_diff' if drake_last['home_name'] == 'Drake Ayala' else 'away_std_point_diff']:.6f}")

# ============================================
# TEST CASE 2: Douglas Shipers (0-14, all losses)
# ============================================
print("\n" + "="*60)
print("TEST CASE 2: Douglas Shipers")
print("="*60)

douglas_last = matches_w_stats.query('home_name == "Douglas Shipers" or away_name == "Douglas Shipers"').iloc[-1]
douglas_col = 'home' if douglas_last['home_name'] == 'Douglas Shipers' else 'away'

print("Expected vs Actual for Douglas Shipers:")
print(f"win_rate:           Expected 0.000000 | Actual {douglas_last[f'{douglas_col}_win_rate']:.6f}")
print(f"pin_count:          Expected 0.000000 | Actual {douglas_last[f'{douglas_col}_pin_count']:.6f}")
print(f"avg_opponent_rank:  Expected 95.076900 | Actual {douglas_last[f'{douglas_col}_avg_opponent_rank']:.6f}")
print(f"avg_point_diff:     Expected -11.000000 | Actual {douglas_last[f'{douglas_col}_avg_point_diff']:.6f}")
print(f"avg_points_scored:  Expected 1.500000 | Actual {douglas_last[f'{douglas_col}_avg_points_scored']:.6f}")
print(f"std_point_diff:     Expected 5.656854 | Actual {douglas_last[f'{douglas_col}_std_point_diff']:.6f}")

# Show his matches to verify
print("\nDouglas Shipers matches (should be 14):")
douglas_matches = matches_w_stats.query('home_name == "Douglas Shipers" or away_name == "Douglas Shipers"')
print(f"Total matches: {len(douglas_matches)}")

# ============================================
# TEST CASE 3: James Conway (ID 2000 - Franklin & Marshall)
# ============================================
print("\n" + "="*60)
print("TEST CASE 3: James Conway (ID 2000 - Franklin & Marshall)")
print("="*60)

james_2000_last = matches_w_stats.query('home_wrestler_id == 2000 or away_wrestler_id == 2000').iloc[-1]
james_col = 'home' if james_2000_last['home_wrestler_id'] == 2000 else 'away'

print("Expected vs Actual for James Conway (ID 2000):")
print(f"win_rate:           Expected 1.000000 | Actual {james_2000_last[f'{james_col}_win_rate']:.6f}")
print(f"bonus_rate:         Expected 0.875000 | Actual {james_2000_last[f'{james_col}_bonus_rate']:.6f}")
print(f"pin_count:          Expected 2.000000 | Actual {james_2000_last[f'{james_col}_pin_count']:.6f}")
print(f"avg_opponent_rank:  Expected 159.5000 | Actual {james_2000_last[f'{james_col}_avg_opponent_rank']:.6f}")
print(f"avg_point_diff:     Expected 13.00000 | Actual {james_2000_last[f'{james_col}_avg_point_diff']:.6f}")
print(f"avg_points_scored:  Expected 15.83333 | Actual {james_2000_last[f'{james_col}_avg_points_scored']:.6f}")
print(f"std_point_diff:     Expected 3.633180 | Actual {james_2000_last[f'{james_col}_std_point_diff']:.6f}")

# Show his matches
print("\nJames Conway (ID 2000) matches:")
james_matches = matches_w_stats.query('home_wrestler_id == 2000 or away_wrestler_id == 2000')
print(f"Total matches: {len(james_matches)}")
for _, row in james_matches.iterrows():
    if row['home_wrestler_id'] == 2000:
        print(f"  Dual {row['dual_id']}: {row['home_name']} vs {row['away_name']} - {'Won' if row['home_win'] else 'Lost'} ({row['win_type']})")
    else:
        print(f"  Dual {row['dual_id']}: {row['away_name']} vs {row['home_name']} - {'Won' if not row['home_win'] else 'Lost'} ({row['win_type']})")

TEST CASE 1: Drake Ayala
Expected vs Actual for Drake Ayala:
win_rate:           Expected 0.600000 | Actual 0.600000
bonus_rate:         Expected 0.466667 | Actual 0.466667
pin_count:          Expected 1.0       | Actual 1.000000
avg_opponent_rank:  Expected 30.866667 | Actual 30.866667
avg_point_diff:     Expected 5.071429  | Actual 5.071429
avg_points_scored:  Expected 10.928571 | Actual 10.928571
std_point_diff:     Expected 8.757101  | Actual 8.757101

TEST CASE 2: Douglas Shipers
Expected vs Actual for Douglas Shipers:
win_rate:           Expected 0.000000 | Actual 0.000000
pin_count:          Expected 0.000000 | Actual 0.000000
avg_opponent_rank:  Expected 95.076900 | Actual 95.076923
avg_point_diff:     Expected -11.000000 | Actual -11.000000
avg_points_scored:  Expected 1.500000 | Actual 1.500000
std_point_diff:     Expected 5.656854 | Actual 5.656854

Douglas Shipers matches (should be 14):
Total matches: 14

TEST CASE 3: James Conway (ID 2000 - Franklin & Marshall)
Expected v

In [38]:
# -- Save data for ML
print(matches_w_stats.isna().sum())

matches_w_stats.to_csv('matches_w_stats_ml.csv', index= False)

dual_id                   0
weight_class              0
event_date                0
home_wrestler_id          0
home_name                 0
home_rank                 0
away_wrestler_id          0
away_name                 0
away_rank                 0
home_win                  0
win_type                  0
Result                    0
home_class                0
home_team_name            0
away_class                0
away_team_name            0
home_matches_wrestled     0
home_win_rate             0
home_loss_rate            0
home_bonus_rate           0
home_pin_count            0
home_avg_opponent_rank    0
home_avg_point_diff       0
home_avg_points_scored    0
home_std_point_diff       0
away_matches_wrestled     0
away_win_rate             0
away_loss_rate            0
away_bonus_rate           0
away_pin_count            0
away_avg_opponent_rank    0
away_avg_point_diff       0
away_avg_points_scored    0
away_std_point_diff       0
dtype: int64


# -- Now compute overall performance across entirety of the season(this function will specifically be for analysis)

In [72]:
def compute_season_statistics(matches_df):
    """
    Compute season-long statistics for each wrestler.
    Used for Tableau analysis - includes all matches (even forfeits/DQs).
    Handles duplicate wrestler names by using wrestler_id as unique identifier.
    """
    print("=" * 80)
    print("COMPUTING SEASON STATISTICS")
    print("=" * 80)
    
    df = matches_df.copy()
    
    # Helper function to extract base win type
    def get_base_win_type(win_type):
        if pd.isna(win_type):
            return None
        wt = str(win_type).upper().strip()
        if "FALL" in wt:
            return "FALL"
        elif "TF" in wt:
            return "TF"
        elif "MD" in wt:
            return "MD"
        elif "DEC" in wt or "SV" in wt or "TB" in wt:
            return "DEC"
        elif "FOR" in wt or "DQ" in wt or "INJ" in wt:
            return "FORFEIT"
        return "OTHER"
    
    # Helper to extract points (only for matches with scores)
    def extract_points(result, win_type):
        if pd.isna(win_type) or pd.isna(result):
            return None, None, False
        
        wt = str(win_type).upper().strip()
        
        # Only score-based matches have points
        if any(x in wt for x in ["DEC", "MD", "TF", "SV", "TB"]):
            try:
                parts = str(result).split()
                scores = [int(x) for x in parts if x.isdigit()]
                if len(scores) >= 2:
                    return scores[0], scores[1], "TF" in wt
            except:
                pass
        
        return None, None, False
    
    # Build wrestler-match records with unique identifier
    records = []
    
    for _, row in df.iterrows():
        # Create unique wrestler keys
        home_key = f"{row['home_wrestler_id']}_{row['home_name']}" if pd.notna(row['home_wrestler_id']) else row['home_name']
        away_key = f"{row['away_wrestler_id']}_{row['away_name']}" if pd.notna(row['away_wrestler_id']) else row['away_name']
        
        base_type = get_base_win_type(row['win_type'])
        score1, score2, is_tf = extract_points(row['Result'], row['win_type'])
        
        # Home wrestler
        home_won = row['home_win']
        records.append({
            'wrestler_id': row['home_wrestler_id'],
            'wrestler_name': row['home_name'],
            'wrestler_key': home_key,
            'team_name': row['home_team_name'],
            'weight_class': row['weight_class'],
            'event_date': row['event_date'],
            'dual_id': row['dual_id'],
            'opponent_name': row['away_name'],
            'opponent_id': row['away_wrestler_id'],
            'opponent_rank': row['away_rank'],
            'win': home_won,
            'win_type': base_type,
            'full_win_type': row['win_type'],
            'result': row['Result'],
            'is_fall': base_type == "FALL",
            'is_forfeit': base_type == "FORFEIT",
            'has_score': score1 is not None,
            'points_scored': score1 if home_won and score1 is not None else (score2 if not home_won and score2 is not None else None),
            'points_allowed': score2 if home_won and score2 is not None else (score1 if not home_won and score1 is not None else None),
            'point_diff': (15 if is_tf and home_won else -15 if is_tf else (score1 - score2 if home_won and score1 is not None else (score2 - score1 if not home_won and score2 is not None else None)))
        })
        
        # Away wrestler
        away_won = not home_won
        records.append({
            'wrestler_id': row['away_wrestler_id'],
            'wrestler_name': row['away_name'],
            'wrestler_key': away_key,
            'team_name': row['away_team_name'],
            'weight_class': row['weight_class'],
            'event_date': row['event_date'],
            'dual_id': row['dual_id'],
            'opponent_name': row['home_name'],
            'opponent_id': row['home_wrestler_id'],
            'opponent_rank': row['home_rank'],
            'win': away_won,
            'win_type': base_type,
            'full_win_type': row['win_type'],
            'result': row['Result'],
            'is_fall': base_type == "FALL",
            'is_forfeit': base_type == "FORFEIT",
            'has_score': score1 is not None,
            'points_scored': score1 if away_won and score1 is not None else (score2 if not away_won and score2 is not None else None),
            'points_allowed': score2 if away_won and score2 is not None else (score1 if not away_won and score1 is not None else None),
            'point_diff': (15 if is_tf and away_won else -15 if is_tf else (score1 - score2 if away_won and score1 is not None else (score2 - score1 if not away_won and score2 is not None else None)))
        })
    
    wrestler_matches = pd.DataFrame(records)
    print(f"✓ Created {len(wrestler_matches)} wrestler-match records")
    
    # Calculate season statistics by wrestler_key (handles duplicates)
    print("\n📊 Aggregating season statistics...")
    
    season_stats = wrestler_matches.groupby(['wrestler_key', 'wrestler_id', 'wrestler_name', 'team_name']).agg(
        # Match counts
        total_matches=('win', 'count'),
        total_wins=('win', 'sum'),
        
        # Win types
        wins_by_fall=('is_fall', lambda x: ((x) & (wrestler_matches.loc[x.index, 'win'])).sum()),
        wins_by_forfeit=('is_forfeit', lambda x: ((x) & (wrestler_matches.loc[x.index, 'win'])).sum()),
        
        # Points (only from matches with scores)
        total_points_scored=('points_scored', 'sum'),
        total_points_allowed=('points_allowed', 'sum'),
        avg_points_scored=('points_scored', lambda x: x.dropna().mean()),
        avg_points_allowed=('points_allowed', lambda x: x.dropna().mean()),
        avg_point_diff=('point_diff', lambda x: x.dropna().mean()),
        std_point_diff=('point_diff', lambda x: x.dropna().std()),
        
        # Opponent quality
        avg_opponent_rank=('opponent_rank', lambda x: x.dropna().mean()),
        
        # Dates
        first_match=('event_date', 'min'),
        last_match=('event_date', 'max')
    ).reset_index()
    
    # Calculate derived statistics
    season_stats['total_losses'] = season_stats['total_matches'] - season_stats['total_wins']
    season_stats['win_rate'] = (season_stats['total_wins'] / season_stats['total_matches'] * 100).round(1)
    season_stats['loss_rate'] = (season_stats['total_losses'] / season_stats['total_matches'] * 100).round(1)
    
    # Bonus wins (TF + MD + FALL)
    # We need to count these properly
    bonus_counts = wrestler_matches[wrestler_matches['win']].groupby('wrestler_key').agg(
        bonus_wins=('win_type', lambda x: sum(t in ['TF', 'MD', 'FALL'] for t in x if pd.notna(t)))
    ).reset_index()
    
    season_stats = season_stats.merge(bonus_counts, on='wrestler_key', how='left')
    season_stats['bonus_wins'] = season_stats['bonus_wins'].fillna(0)
    season_stats['bonus_rate'] = (season_stats['bonus_wins'] / season_stats['total_matches'] * 100).round(1)
    
    # Create win type distribution columns
    win_type_dist = wrestler_matches[wrestler_matches['win']].groupby(['wrestler_key', 'win_type']).size().unstack(fill_value=0).reset_index()
    
    for col in ['DEC', 'MD', 'TF', 'FALL', 'FORFEIT', 'OTHER']:
        if col not in win_type_dist.columns:
            win_type_dist[col] = 0
    
    season_stats = season_stats.merge(
        win_type_dist[['wrestler_key', 'DEC', 'MD', 'TF', 'FALL', 'FORFEIT']], 
        on='wrestler_key', 
        how='left'
    )
    
    # Format numeric columns
    season_stats['avg_opponent_rank'] = season_stats['avg_opponent_rank'].round(2)
    season_stats['avg_points_scored'] = season_stats['avg_points_scored'].round(2)
    season_stats['avg_points_allowed'] = season_stats['avg_points_allowed'].round(2)
    season_stats['avg_point_diff'] = season_stats['avg_point_diff'].round(2)
    season_stats['std_point_diff'] = season_stats['std_point_diff'].round(2)
    
    # Create record string
    season_stats['record'] = season_stats['total_wins'].astype(int).astype(str) + '-' + \
                             season_stats['total_losses'].astype(int).astype(str)
    
    # Select final columns for Tableau
    final_columns = [
        'wrestler_id', 'wrestler_name', 'team_name',
        'total_matches', 'record', 'total_wins', 'total_losses',
        'win_rate', 'loss_rate',
        'bonus_wins', 'bonus_rate',
        'DEC', 'MD', 'TF', 'FALL', 'FORFEIT',
        'avg_opponent_rank',
        'avg_points_scored', 'avg_points_allowed', 'avg_point_diff', 'std_point_diff',
        'first_match', 'last_match'
    ]
    
    season_stats = season_stats[final_columns]
    season_stats = season_stats.sort_values('total_matches', ascending=False).reset_index(drop=True)
    
    print(f"✓ Aggregated stats for {len(season_stats)} unique wrestlers")
    
    return season_stats, wrestler_matches

# Usage:
season_stats, wrestler_matches_long = compute_season_statistics(matches)
# 
# # For Tableau, export both:
# season_stats.to_csv('wrestler_season_stats.csv', index=False)
# wrestler_matches_long.to_csv('wrestler_matches_long.csv', index=False)

COMPUTING SEASON STATISTICS
✓ Created 11680 wrestler-match records

📊 Aggregating season statistics...
✓ Aggregated stats for 1512 unique wrestlers


In [71]:
# ============================================
# TEST CASE 1: Drake Ayala
# ============================================
print("\n" + "="*80)
print("TEST CASE 1: DRAKE AYALA")
print("="*80)

drake = season_stats[season_stats['wrestler_name'] == 'Drake Ayala']

if len(drake) > 0:
    drake = drake.iloc[0]
    
    print(f"\n✓ Found Drake Ayala (ID: {drake['wrestler_id']})")
    print(f"Team: {drake['team_name']}")
    
    expected = {
        'total_matches': 16,
        'total_wins': 9,
        'bonus_wins': 7,
        'TF': 5,
        'FALL': 1,
        'MD': 1,
        'DEC': 2,
        'avg_opponent_rank': 29.25,
        'avg_points_scored': 10.4,
        'avg_point_diff': 3.7333,
        'std_point_diff': 9.902862
    }
    
    print(f"\n{'Metric':<25} {'Expected':<15} {'Actual':<15} {'Status'}")
    print("─" * 70)
    
    results = {}
    for metric, exp_val in expected.items():
        actual_val = drake[metric]
        
        if isinstance(exp_val, int):
            match = actual_val == exp_val
            tolerance = ""
        else:
            match = abs(actual_val - exp_val) < 0.1
            tolerance = f" (±0.1)"
        
        status = "✓" if match else "✗"
        results[metric] = match
        
        print(f"{metric:<25} {str(exp_val):<15} {str(actual_val):<15} {status}")
    
    passed = sum(1 for v in results.values() if v)
    total = len(results)
    print("─" * 70)
    print(f"Result: {passed}/{total} tests passed")
else:
    print("\n❌ Drake Ayala not found in data!")

# ============================================
# TEST CASE 2: Nathan Taylor (ID 2001)
# ============================================
print("\n" + "="*80)
print("TEST CASE 2: NATHAN TAYLOR (ID 2001)")
print("="*80)

nathan = season_stats[season_stats['wrestler_id'] == 2001.0]

if len(nathan) > 0:
    nathan = nathan.iloc[0]
    
    print(f"\n✓ Found Nathan Taylor (ID: 2001)")
    print(f"Team: {nathan['team_name']}")
    
    expected = {
        'total_matches': 3,
        'total_wins': 3,
        'bonus_wins': 2,
        'TF': 1,
        'MD': 1,
        'DEC': 1,
        'avg_opponent_rank': 151,
        'avg_points_scored': 12.3333,
        'avg_point_diff': 9,
        'std_point_diff': 5.567764
    }
    
    print(f"\n{'Metric':<25} {'Expected':<15} {'Actual':<15} {'Status'}")
    print("─" * 70)
    
    results = {}
    for metric, exp_val in expected.items():
        actual_val = nathan[metric]
        
        if isinstance(exp_val, int):
            match = actual_val == exp_val
            tolerance = ""
        else:
            match = abs(actual_val - exp_val) < 0.1
            tolerance = f" (±0.1)"
        
        status = "✓" if match else "✗"
        results[metric] = match
        
        print(f"{metric:<25} {str(exp_val):<15} {str(actual_val):<15} {status}")
    
    passed = sum(1 for v in results.values() if v)
    total = len(results)
    print("─" * 70)
    print(f"Result: {passed}/{total} tests passed")
    
    # Show his matches for verification
    print("\nNathan Taylor (ID 2001) matches:")
    nathan_matches = matches.query('home_wrestler_id == 2001 or away_wrestler_id == 2001')
    for _, row in nathan_matches.iterrows():
        if row['home_wrestler_id'] == 2001:
            print(f"  Dual {row['dual_id']}: {row['home_name']} vs {row['away_name']} - {row['win_type']}")
        else:
            print(f"  Dual {row['dual_id']}: {row['away_name']} vs {row['home_name']} - {row['win_type']}")
else:
    print("\n❌ Nathan Taylor (ID 2001) not found in data!")

# ============================================
# TEST CASE 3: Angelo Ferrari
# ============================================
print("\n" + "="*80)
print("TEST CASE 3: ANGELO FERRARI")
print("="*80)

angelo = season_stats[season_stats['wrestler_name'] == 'Angelo Ferrari']

if len(angelo) > 0:
    angelo = angelo.iloc[0]
    
    print(f"\n✓ Found Angelo Ferrari (ID: {angelo['wrestler_id']})")
    print(f"Team: {angelo['team_name']}")
    
    expected = {
        'total_matches': 10,
        'total_wins': 9,
        'bonus_wins': 3,
        'TF': 3,
        'DEC': 6,
        'avg_opponent_rank': 35,
        'avg_points_scored': 8.4,
        'avg_point_diff': 5.8,
        'std_point_diff': 6.494442
    }
    
    print(f"\n{'Metric':<25} {'Expected':<15} {'Actual':<15} {'Status'}")
    print("─" * 70)
    
    results = {}
    for metric, exp_val in expected.items():
        if metric in ['TB', 'SV']:  # These might be combined into DEC
            continue
            
        actual_val = angelo[metric]
        
        if isinstance(exp_val, int):
            match = actual_val == exp_val
        else:
            match = abs(actual_val - exp_val) < 0.1
        
        status = "✓" if match else "✗"
        results[metric] = match
        
        print(f"{metric:<25} {str(exp_val):<15} {str(actual_val):<15} {status}")
    
    # Check TB and SV if your function tracks them separately
    if 'TB' in angelo.index and 'SV' in angelo.index:
        tb_match = angelo['TB'] == 1
        sv_match = angelo['SV'] == 1
        print(f"{'TB':<25} {'1':<15} {str(angelo['TB']):<15} {'✓' if tb_match else '✗'}")
        print(f"{'SV':<25} {'1':<15} {str(angelo['SV']):<15} {'✓' if sv_match else '✗'}")
        results['TB'] = tb_match
        results['SV'] = sv_match
    
    passed = sum(1 for v in results.values() if v)
    total = len(results)
    print("─" * 70)
    print(f"Result: {passed}/{total} tests passed")
    
    # Show his matches for verification
    print("\nAngelo Ferrari matches:")
    angelo_matches = matches.query('home_name == "Angelo Ferrari" or away_name == "Angelo Ferrari"').sort_values('event_date')
    for idx, row in angelo_matches.iterrows():
        if row['home_name'] == 'Angelo Ferrari':
            print(f"  Dual {row['dual_id']}: {row['home_name']} vs {row['away_name']} - {row['win_type']}")
        else:
            print(f"  Dual {row['dual_id']}: {row['away_name']} vs {row['home_name']} - {row['win_type']}")
else:
    print("\n❌ Angelo Ferrari not found in data!")

# ============================================
# SUMMARY
# ============================================
print("\n" + "="*80)
print("TEST SUMMARY")
print("="*80)

summary_data = [
    ['Drake Ayala', '✓' if len(season_stats[season_stats['wrestler_name'] == 'Drake Ayala']) > 0 else '✗'],
    ['Nathan Taylor (ID 2001)', '✓' if len(season_stats[season_stats['wrestler_id'] == 2001.0]) > 0 else '✗'],
    ['Angelo Ferrari', '✓' if len(season_stats[season_stats['wrestler_name'] == 'Angelo Ferrari']) > 0 else '✗']
]

print(f"\n{'Wrestler':<25} {'Found':<10}")
print("─" * 35)
for row in summary_data:
    print(f"{row[0]:<25} {row[1]:<10}")


TEST CASE 1: DRAKE AYALA

✓ Found Drake Ayala (ID: 196.0)
Team: Iowa

Metric                    Expected        Actual          Status
──────────────────────────────────────────────────────────────────────
total_matches             16              16              ✓
total_wins                9               9               ✓
bonus_wins                7               7.0             ✓
TF                        5               5.0             ✓
FALL                      1               1.0             ✓
MD                        1               1.0             ✓
DEC                       2               2.0             ✓
avg_opponent_rank         29.25           29.25           ✓
avg_points_scored         10.4            10.4            ✓
avg_point_diff            3.7333          3.73            ✓
std_point_diff            9.902862        9.9             ✓
──────────────────────────────────────────────────────────────────────
Result: 11/11 tests passed

TEST CASE 2: NATHAN TAYLOR (ID 200

# Format data for tableau

In [73]:
season_stats = season_stats.fillna(0) 
season_stats[["wrestler_id", "wrestler_name", "team_name", "bonus_rate", "total_matches", "win_rate", "FALL", "TF", "avg_point_diff", "avg_points_scored"]]

,wrestler_id,wrestler_name,team_name,bonus_rate,total_matches,win_rate,FALL,TF,avg_point_diff,avg_points_scored
0,85.0,William Morrow,Bloomsburg,4.5,22,13.6,0.0,1.0,-2.77,2.68
1,89.0,Mason Rebuck,Bloomsburg,50.0,20,75.0,6.0,2.0,3.92,8.17
2,22.0,Kyle Waterman,Drexel,30.0,20,50.0,2.0,3.0,1.19,8.50
3,23.0,Jordan Soriano,Drexel,35.0,20,70.0,5.0,0.0,2.23,7.31
4,100.0,Ryan Bolletino,Campbell,15.0,20,30.0,1.0,1.0,-2.46,5.08
...,...,...,...,...,...,...,...,...,...,...
1507,1241.0,Jonathan Chesser,The Citadel,100.0,1,100.0,0.0,1.0,15.00,16.00
1508,1503.0,Parker Ferrell,Virginia Tech,0.0,1,0.0,0.0,0.0,-8.00,0.00
1509,1504.0,Nick Fea,North Carolina,0.0,1,0.0,0.0,0.0,-3.00,2.00
1510,1505.0,Brockton Borelli,Stanford,0.0,1,0.0,0.0,0.0,-7.00,2.00


In [74]:
season_stats.query('wrestler_name.str.contains("Taylor")', engine='python')

,wrestler_id,wrestler_name,team_name,total_matches,record,total_wins,total_losses,win_rate,loss_rate,bonus_wins,...,TF,FALL,FORFEIT,avg_opponent_rank,avg_points_scored,avg_points_allowed,avg_point_diff,std_point_diff,first_match,last_match
48,219.0,Antrell Taylor,Nebraska,18,16-2,16,2,88.9,11.1,9.0,...,5.0,2.0,1.0,31.65,9.87,4.00,5.67,8.72,2025-11-07,2026-02-21
352,168.0,Nathan Taylor,Lehigh,13,10-3,10,3,76.9,23.1,2.0,...,2.0,0.0,2.0,48.50,6.91,3.45,3.45,6.96,2025-12-05,2026-02-21
549,329.0,Shawn Taylor,West Virginia,10,2-8,2,8,20.0,80.0,0.0,...,0.0,0.0,0.0,66.50,4.88,7.88,-2.88,6.45,2025-11-08,2026-02-06
1052,2001.0,Nathan Taylor,Pennsylvania,3,3-0,3,0,100.0,0.0,2.0,...,1.0,0.0,0.0,151.00,12.33,2.00,9.00,5.57,2025-11-15,2026-01-11
1158,833.0,Kevin Taylor,Sacred Heart,2,0-2,0,2,0.0,100.0,0.0,...,0.0,0.0,0.0,61.00,2.50,15.50,-12.50,3.54,2025-12-20,2026-01-18
1373,1314.0,Hunter Taylor,Oregon State,1,1-0,1,0,100.0,0.0,1.0,...,1.0,0.0,0.0,243.00,19.00,3.00,15.00,0.00,2026-01-16,2026-01-16


In [42]:
# -- Subset for wrestler with only 7 or more matches
season_stats_copy = season_stats.query('total_matches > 6').copy()

In [43]:
season_stats_wrestlers = pd.melt(
    season_stats_copy[
        ["wrestler_id", "wrestler_name", "team_name",
         "bonus_rate", "total_matches", "win_rate",
         "FALL", "TF", "avg_point_diff", "avg_points_scored"]
    ],
    id_vars=["wrestler_id", "wrestler_name", "team_name"],
    value_vars=["bonus_rate", "total_matches", "win_rate",
                "FALL", "TF", "avg_point_diff", "avg_points_scored"],
    var_name="stat",
    value_name="value"
)

season_stats_wrestlers = season_stats_wrestlers
season_stats_wrestlers

,wrestler_id,wrestler_name,team_name,stat,value
0,85.0,William Morrow,Bloomsburg,bonus_rate,4.50
1,89.0,Mason Rebuck,Bloomsburg,bonus_rate,50.00
2,22.0,Kyle Waterman,Drexel,bonus_rate,30.00
3,23.0,Jordan Soriano,Drexel,bonus_rate,35.00
4,100.0,Ryan Bolletino,Campbell,bonus_rate,15.00
...,...,...,...,...,...
5630,903.0,Teage Calvin,American,avg_points_scored,2.14
5631,93.0,CJ Walrath,Northern Iowa,avg_points_scored,6.86
5632,1343.0,Liam Carlin,Pennsylvania,avg_points_scored,3.71
5633,309.0,Mac Church,Virginia Tech,avg_points_scored,5.57


# -- Get wrestlers Primary Weight

In [75]:
all_wrestlers

,Team,Weight,Name,Class,Record,Rank,Last,First
0,Air Force,125,"#28 Owens, TuckerST",SR,19 - 8,28,Owens,Tucker
1,Air Force,125,"#157 Gonzalez, Nick",SR,0 - 1,157,Gonzalez,Nick
2,Air Force,125,"#181 Tocci, Nico",JR,0 - 2,181,Tocci,Nico
3,Air Force,125,"#212 Bohnsack, Brayden",FR,0 - 0,212,Bohnsack,Brayden
4,Air Force,133,"#121 Caprella, GavinST",JR,9 - 12,121,Caprella,Gavin
...,...,...,...,...,...,...,...,...
2640,Wyoming,197,"#3 Novak, JoeyST",JR,14 - 2,3,Novak,Joey
2641,Wyoming,197,"#88 Henry, GunnerRS",FR,6 - 3,88,Henry,Gunner
2642,Wyoming,285,"#11 Carroll, ChristianST",SO,16 - 4,11,Carroll,Christian
2643,Wyoming,285,"#182 McBride, Winston",SO,1 - 0,182,McBride,Winston


In [76]:
STATUS_CODES = ["ST", "RS", "RSFR", "RSSO", "RSJR"]

def clean_name(name_str):
    # Extract rank
    rank_match = re.search(r"#(\d+)", name_str)
    rank = int(rank_match.group(1)) if rank_match else None

    # Remove rank
    name_only = re.sub(r"#\d+\s*", "", name_str)

    # Remove trailing known status ONLY if attached
    for status in STATUS_CODES:
        if name_only.endswith(status):
            name_only = name_only[:-len(status)]
            break

    name_only = name_only.strip()

    # Split last and first
    if "," in name_only:
        last, first = name_only.split(",", 1)
        return rank, last.strip(), first.strip()

    return rank, None, None
#🚀 Apply It To Your DataFrame
all_wrestlers[["Rank", "Last", "First"]] = all_wrestlers["Name"].apply(
    lambda x: pd.Series(clean_name(x))
)

all_wrestlers_info = all_wrestlers.copy()
all_wrestlers_info['name'] = all_wrestlers_info['First'] + " " + all_wrestlers_info['Last']

all_wrestlers_info


,Team,Weight,Name,Class,Record,Rank,Last,First,name
0,Air Force,125,"#28 Owens, TuckerST",SR,19 - 8,28,Owens,Tucker,Tucker Owens
1,Air Force,125,"#157 Gonzalez, Nick",SR,0 - 1,157,Gonzalez,Nick,Nick Gonzalez
2,Air Force,125,"#181 Tocci, Nico",JR,0 - 2,181,Tocci,Nico,Nico Tocci
3,Air Force,125,"#212 Bohnsack, Brayden",FR,0 - 0,212,Bohnsack,Brayden,Brayden Bohnsack
4,Air Force,133,"#121 Caprella, GavinST",JR,9 - 12,121,Caprella,Gavin,Gavin Caprella
...,...,...,...,...,...,...,...,...,...
2640,Wyoming,197,"#3 Novak, JoeyST",JR,14 - 2,3,Novak,Joey,Joey Novak
2641,Wyoming,197,"#88 Henry, GunnerRS",FR,6 - 3,88,Henry,Gunner,Gunner Henry
2642,Wyoming,285,"#11 Carroll, ChristianST",SO,16 - 4,11,Carroll,Christian,Christian Carroll
2643,Wyoming,285,"#182 McBride, Winston",SO,1 - 0,182,McBride,Winston,Winston McBride


In [77]:
season_stats.columns

Index(['wrestler_id', 'wrestler_name', 'team_name', 'total_matches', 'record',
       'total_wins', 'total_losses', 'win_rate', 'loss_rate', 'bonus_wins',
       'bonus_rate', 'DEC', 'MD', 'TF', 'FALL', 'FORFEIT', 'avg_opponent_rank',
       'avg_points_scored', 'avg_points_allowed', 'avg_point_diff',
       'std_point_diff', 'first_match', 'last_match'],
      dtype='object')

In [55]:
season_stats_wrestlers = season_stats_wrestlers.merge(
    all_wrestlers_info[['name', 'Team', 'Weight']],
    left_on=['wrestler_name', 'team_name'],
    right_on=['name', 'Team'],
    how='left'
)

# Optional: drop duplicate join columns
season_stats_wrestlers = season_stats_wrestlers.drop(columns=['name', 'Team'])

season_stats_wrestlers

,wrestler_id,wrestler_name,team_name,stat,value,Weight
0,85.0,William Morrow,Bloomsburg,bonus_rate,4.50,157.0
1,89.0,Mason Rebuck,Bloomsburg,bonus_rate,50.00,285.0
2,22.0,Kyle Waterman,Drexel,bonus_rate,30.00,133.0
3,23.0,Jordan Soriano,Drexel,bonus_rate,35.00,141.0
4,100.0,Ryan Bolletino,Campbell,bonus_rate,15.00,174.0
...,...,...,...,...,...,...
5630,903.0,Teage Calvin,American,avg_points_scored,2.14,197.0
5631,93.0,CJ Walrath,Northern Iowa,avg_points_scored,6.86,184.0
5632,1343.0,Liam Carlin,Pennsylvania,avg_points_scored,3.71,184.0
5633,309.0,Mac Church,Virginia Tech,avg_points_scored,5.57,165.0


In [59]:
# -- Same problem as early
season_stats_wrestlers[season_stats_wrestlers.isna().any(axis=1)]

,wrestler_id,wrestler_name,team_name,stat,value,Weight
415,341.0,Jesse Perez,Northwestern,bonus_rate,33.30,NaN
1220,341.0,Jesse Perez,Northwestern,total_matches,12.00,NaN
2025,341.0,Jesse Perez,Northwestern,win_rate,50.00,NaN
2830,341.0,Jesse Perez,Northwestern,FALL,0.00,NaN
3635,341.0,Jesse Perez,Northwestern,TF,0.00,NaN
4440,341.0,Jesse Perez,Northwestern,avg_point_diff,-2.09,NaN
5245,341.0,Jesse Perez,Northwestern,avg_points_scored,8.64,NaN


In [60]:
all_wrestlers_info.query('name == "JD Perez"')

,Team,Weight,Name,Class,Record,Rank,Last,First,name
1780,Northwestern,184,"#60 Perez, JDST",SR,9 - 13,60,Perez,JD,JD Perez


In [61]:
season_stats_wrestlers.loc[
    (season_stats_wrestlers['wrestler_id'] == 341) &
    (season_stats_wrestlers['team_name'] == 'Northwestern'),
    'Weight'
] = 184



In [70]:
season_stats_wrestlers.query('wrestler_name == "Nathan Taylor"')

,wrestler_id,wrestler_name,team_name,stat,value,Weight
352,168.0,Nathan Taylor,Lehigh,bonus_rate,15.40,285.0
1157,168.0,Nathan Taylor,Lehigh,total_matches,13.00,285.0
1962,168.0,Nathan Taylor,Lehigh,win_rate,76.90,285.0
2767,168.0,Nathan Taylor,Lehigh,FALL,0.00,285.0
3572,168.0,Nathan Taylor,Lehigh,TF,2.00,285.0
4377,168.0,Nathan Taylor,Lehigh,avg_point_diff,3.45,285.0
5182,168.0,Nathan Taylor,Lehigh,avg_points_scored,6.91,285.0


In [81]:
season_stats_wrestlers["Wrestler"] = season_stats_wrestlers["wrestler_name"] + " - " + season_stats_wrestlers["team_name"]

season_stats_wrestlers.to_csv('wrestler_stats_season.csv', index=False)

season_stats_wrestlers

,wrestler_id,wrestler_name,team_name,stat,value,Weight,Wrestler
0,85.0,William Morrow,Bloomsburg,bonus_rate,4.50,157.0,William Morrow - Bloomsburg
1,89.0,Mason Rebuck,Bloomsburg,bonus_rate,50.00,285.0,Mason Rebuck - Bloomsburg
2,22.0,Kyle Waterman,Drexel,bonus_rate,30.00,133.0,Kyle Waterman - Drexel
3,23.0,Jordan Soriano,Drexel,bonus_rate,35.00,141.0,Jordan Soriano - Drexel
4,100.0,Ryan Bolletino,Campbell,bonus_rate,15.00,174.0,Ryan Bolletino - Campbell
...,...,...,...,...,...,...,...
5630,903.0,Teage Calvin,American,avg_points_scored,2.14,197.0,Teage Calvin - American
5631,93.0,CJ Walrath,Northern Iowa,avg_points_scored,6.86,184.0,CJ Walrath - Northern Iowa
5632,1343.0,Liam Carlin,Pennsylvania,avg_points_scored,3.71,184.0,Liam Carlin - Pennsylvania
5633,309.0,Mac Church,Virginia Tech,avg_points_scored,5.57,165.0,Mac Church - Virginia Tech


In [82]:
matches.query('home_name == "TK Davis"')

,dual_id,weight_class,event_date,home_wrestler_id,home_name,home_rank,away_wrestler_id,away_name,away_rank,home_win,win_type,Result,home_class,home_team_name,away_class,away_team_name
641,240,133,2025-11-15,985.0,TK Davis,19.0,345.0,Braxton Brown,17.0,True,WDEC,WDEC 6 - 3,SO,Gardner-Webb,SR,Maryland
676,241,133,2025-11-15,985.0,TK Davis,19.0,881.0,Gavin Caprella,76.0,True,WDEC,WDEC 7 - 3,SO,Gardner-Webb,JR,Air Force
1239,161,133,2025-12-03,985.0,TK Davis,19.0,1080.0,John King,194.0,True,WTF,WTF5 19 - 3 3:20,SO,Gardner-Webb,RSFR,Duke
1343,156,133,2025-12-05,985.0,TK Davis,19.0,1044.0,Andrew Hampton,149.0,True,WTF,WTF5 24 - 6 6:34,SO,Gardner-Webb,SR,Michigan State
3145,347,133,2026-01-16,985.0,TK Davis,18.0,1330.0,Ethan Uhorchuk,61.0,True,WMD,WMD 17 - 6,SO,Gardner-Webb,FR,Chattanooga
3659,388,133,2026-01-23,985.0,TK Davis,15.0,685.0,Joseph Fischer,82.0,True,WMD,WMD 14 - 3,SO,Gardner-Webb,RSSR,Clarion
3965,432,141,2026-01-29,985.0,TK Davis,15.0,800.0,Marley Washington,224.0,True,WMD,WMD 20 - 6,SO,Gardner-Webb,RSFR,Davidson
4670,527,133,2026-02-06,985.0,TK Davis,14.0,734.0,Teegan Vasquez,134.0,True,WMD,WMD 16 - 7,SO,Gardner-Webb,FR,The Citadel
5361,455,133,2026-02-15,985.0,TK Davis,14.0,1332.0,Cody Tanner,216.0,True,WTF,WTF5 17 - 2 3:41,SO,Gardner-Webb,SO,VMI
5500,583,133,2026-02-19,985.0,TK Davis,14.0,1290.0,Gunner Chambers,166.0,True,WTF,WTF5 21 - 5 4:43,SO,Gardner-Webb,SO,George Mason
